# 02b — Derivatives and optimization — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/02.md) · [Course map](../PLAN.md) · [Project](../../projects/stage02/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Approximate and derive gradients
- Implement gradient descent
- Diagnose learning-rate failures


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


Course root: /Users/sureshkarki/Programming/GitExp/personal/python/learning


## Start here: change, derivatives, and learning

**Prerequisites:** substitute values into a function, square a number, and read a line's slope. Review [02a's concepts](../concepts/02a.md) or [the arithmetic bridge](../MATH_START_HERE.md) first. Calculus here answers a practical question: which small change to a model would reduce its error?

### 1. Average change before instantaneous change

Suppose distance changes from 10 to 18 metres during 2 seconds. Average speed is change in distance divided by change in time: $(18-10)/2=4$ metres/second. The symbol $\Delta$ means “change,” so $\Delta y/\Delta x$ is a change ratio.

For $f(x)=x^2$, moving from $x=2$ to $x=3$ gives average slope $(9-4)/(3-2)=5$. Moving only from 2 to 2.1 gives $(4.41-4)/0.1=4.1$. Smaller steps approach 4. The **derivative** at 2 is this limiting slope, written $f'(2)=4$. A limit describes what happens as the step approaches zero; it does not mean dividing by zero.

### 2. Derive one rule instead of memorizing it

For a small nonzero step $h$:

$$\frac{(x+h)^2-x^2}{h}=\frac{x^2+2xh+h^2-x^2}{h}=2x+h.$$

As $h$ approaches zero, this approaches $2x$. Therefore the derivative of $x^2$ is $2x$. A constant has derivative zero because it does not change. A line $ax+b$ has derivative $a$. These rules let us differentiate the squared error used below.

The **chain rule** handles nested operations. For $g(w)=(3w-5)^2$, first differentiate the outside square, giving $2(3w-5)$, then multiply by the inside derivative 3: $g'(w)=6(3w-5)$. Omitting the inside factor gives the wrong sensitivity.

### 3. Predictions, targets, and loss

A one-feature model predicts $\hat y=wx+b$. The hat means “prediction”; $y$ is the observed target. $w$ is the weight/slope and $b$ is the bias/intercept. For an input $x=2$, weight $w=1$, bias $b=0$, the prediction is 2. If the observed target is 5, the **residual** is prediction minus target: $r=2-5=-3$. Its squared error is $r^2=9$.

For $n$ examples, the **mean squared error** is $L=(1/n)\sum_{i=1}^{n}(wx_i+b-y_i)^2$. Sigma means “add these terms”; subscript $i$ selects one example. Compute every residual, square each one, add the squares, and divide by the number of examples. Squaring prevents positive and negative errors from cancelling and penalizes large errors more strongly.

### 4. Partial derivatives and the gradient

Loss depends on both $w$ and $b$. A **partial derivative** changes one parameter while holding the other fixed. Apply the chain rule to each example:

- Changing $w$ changes the prediction by $x$, so $\partial L/\partial w=(2/n)\sum x_i r_i$.
- Changing $b$ changes the prediction by 1, so $\partial L/\partial b=(2/n)\sum r_i$.

The **gradient** is the ordered collection of these partial derivatives. It points toward the steepest local increase under the usual Euclidean geometry. To reduce loss, step in the opposite direction.

### 5. Perform one learning step by hand

Use inputs `[0,1]`, targets `[1,3]`, and initial $w=b=0$. Predictions are `[0,0]`, residuals `[-1,-3]`, and loss $(1+9)/2=5$.

The weight derivative is $2(0(-1)+1(-3))/2=-3$. The bias derivative is $2(-1-3)/2=-4$. With learning rate $\alpha=0.1$:

$$w_{new}=0-0.1(-3)=0.3,\qquad b_{new}=0-0.1(-4)=0.4.$$

Use the **old** parameters to calculate both gradients, then update. New predictions are `[0.4,0.7]`; new loss is $((-.6)^2+(-2.3)^2)/2=2.825$, lower than 5. Learning is repeated arithmetic of this form. A negative gradient makes the parameter increase because subtracting a negative adds.

### 6. Learning rate and regularization

The learning rate sets the step size. A very small rate progresses slowly; a very large rate can jump across a valley and increase loss. For $L(w)=w^2$, the update is $w_{new}=(1-2\alpha)w$. At $\alpha=.1$, magnitude shrinks by a factor .8. At $\alpha=1.2$, the factor is -1.4: the sign alternates and the magnitude grows. The stable rate depends on the curvature and feature scale, not a universal magic value.

**L2 regularization** adds $\lambda w^2$ to the loss. Lambda is a nonnegative penalty strength. Its weight derivative adds $2\lambda w$; this convention does not penalize the intercept. The fitted slope may shrink, so a regularized optimum need not exactly match an unpenalized line. Distinguish the learning rate (how you search) from the penalty strength (which objective you minimize).

### 7. Check calculus numerically

The central difference $[L(w+h)-L(w-h)]/(2h)$ estimates the derivative without using its formula. Try $h=10^{-5}$ and compare it to the analytical derivative. Restore the original parameter after each perturbation. Too large a step approximates poorly; an extremely tiny step loses precision when subtracting nearly equal floating-point numbers. Compare with tolerance, not exact `==`.

### Self-check

1. What is the derivative of $x^2$ at $x=-2$?
2. If a gradient is +6 and the learning rate is .05, how does the parameter change?
3. With predictions `[2,4]` and targets `[1,2]`, what is MSE?
4. Why can a correct gradient implementation still fail to train?

<details><summary>Answers and reasoning</summary>

1. -4. The function locally decreases as x moves right at -2.
2. Subtract .3 from its old value.
3. Residuals `[1,2]`; squared errors `[1,4]`; average 2.5.
4. An unstable rate, unsuitable scale, wrong update order, or unsuitable model can still prevent learning. A gradient check diagnoses derivatives, not the whole pipeline.

</details>


## Lesson

A derivative is a local rate of change, defined as the limit of a difference quotient. A partial derivative changes one input while holding others fixed. A gradient collects partial derivatives. The chain rule propagates derivatives through composed functions, which makes backpropagation possible.

For mean squared error L(w)=mean((wx-y)^2), dL/dw=mean(2x(wx-y)). Gradient descent updates w ← w-η∇L. The minus sign moves downhill locally, and η is the learning rate. Too large a rate can diverge; too small a rate can stall. A finite-difference check compares your analytical gradient to (L(w+h)-L(w-h))/(2h). Very tiny h suffers cancellation.

Regularization adds a preference to the objective: L2 adds λw² and gradient 2λw. Nonconvex objectives may have local minima and saddle points; a decreasing training loss alone says nothing about generalization.


## Worked example

Predict the output before running. Change one input and explain the result.


In [8]:
def loss(w):
    return (w - 3) ** 2
w = 0.0
for _ in range(5):
    w -= 0.1 * 2 * (w - 3)
    print(round(w, 4), round(loss(w), 4))


0.6 5.76
1.08 3.6864
1.464 2.3593
1.7712 1.5099
2.017 0.9664


## Exercise 1: Numerical derivative

Approximate f prime at x using a central difference with h > 0.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def derivative(f, x, h=1e-5):
    raise NotImplementedError


In [ ]:
assert abs(derivative(lambda x: x*x, 3)-6) < 1e-6
expect_error(ValueError, lambda: derivative(lambda x: x, 0, 0))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Evaluate on both sides of x.

</details>


## Exercise 2: Linear gradient

Return the MSE gradient for scalar w on equally sized nonempty xs, ys.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def mse_gradient(w, xs, ys):
    raise NotImplementedError


In [ ]:
xs, ys = [1, 2, 3], [2, 4, 6]
f = lambda w: sum((w*x-y)**2 for x,y in zip(xs,ys))/len(xs)
assert abs(mse_gradient(1, xs, ys)-derivative(f, 1)) < 1e-6
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Differentiate each residual squared, then average.

</details>


## Exercise 3: Fit one weight

Start at w=0, apply steps gradient updates, and return w. Defaults should fit y=2x.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def fit_weight(xs, ys, lr=0.05, steps=100):
    raise NotImplementedError


In [ ]:
assert abs(fit_weight([1, 2, 3], [2, 4, 6])-2) < 1e-6
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Recompute the gradient after every update.

</details>


## Independent transfer

Add an intercept, L2 regularization, and loss history. Compare rates 0.001, 0.05, and 1.0. Explain divergence from the update equation; save a loss plot.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [02b interface and acceptance cases](../assignments/02b.md).
2. Copy the [blank starter](../assignments/starters/02b.py) to `work/assignments/02b.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 02b --solution work/assignments/02b.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/02b.md#02b-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 02b --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. What does the chain rule do?

2. Does low training loss imply good predictions?


## Reading and review

- [Primary reference 1](https://d2l.ai/chapter_preliminaries/calculus.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
